# Aimran Data Science Lab — Enterprise Demo

This notebook walks through the core capabilities of the Aimran platform:

| Section | What it covers |
|---------|----------------|
| 1 | **AutoML** — automatic model selection |
| 2 | **Experiment Tracking** — log & compare runs |
| 3 | **Model Explainability** — feature importance |
| 4 | **Data Drift Detection** — monitor distribution shift |
| 5 | **Data Quality** — auto-profiling & rules engine |

> Run cells top-to-bottom. If an optional module is not yet implemented,
> a stub/fallback is used so the rest of the notebook still executes.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ── Core platform imports ──────────────────────────────────────────────────
from src.models.random_forest import RandomForestModel
from src.models.registry import register_model, list_models, get_model

# ── Optional / upcoming module imports (graceful fallback) ─────────────────
try:
    from src.automl.engine import AutoMLEngine, AutoMLConfig
    _AUTOML_AVAILABLE = True
except ImportError:
    _AUTOML_AVAILABLE = False
    print('ℹ️  src.automl not yet implemented — using built-in stub for demo.')

try:
    from src.experiments.manager import ExperimentManager
    _EXPERIMENTS_AVAILABLE = True
except ImportError:
    _EXPERIMENTS_AVAILABLE = False
    print('ℹ️  src.experiments not yet implemented — using built-in stub for demo.')

try:
    from src.explainability.shap_engine import ExplainabilityEngine
    _EXPLAIN_AVAILABLE = True
except ImportError:
    _EXPLAIN_AVAILABLE = False
    print('ℹ️  src.explainability not yet implemented — using built-in stub for demo.')

try:
    from src.monitoring.drift_detector import DataDriftDetector
    _DRIFT_AVAILABLE = True
except ImportError:
    _DRIFT_AVAILABLE = False
    print('ℹ️  src.monitoring.drift_detector not yet implemented — using built-in stub for demo.')

try:
    from src.data.quality import DataQualityEngine
    _QUALITY_AVAILABLE = True
except ImportError:
    _QUALITY_AVAILABLE = False
    print('ℹ️  src.data.quality not yet implemented — using built-in stub for demo.')

print('\n✅ Imports successful')

## 1. AutoML — Automatic Model Selection

The `AutoMLEngine` searches a configurable model space and returns
a ranked leaderboard together with the best model and its rationale.

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features, {y.nunique()} classes')

# ── AutoML ────────────────────────────────────────────────────────────────
if _AUTOML_AVAILABLE:
    config = AutoMLConfig(
        task='classification',
        time_budget=60,
        n_trials=10,
        metric='accuracy',
    )
    engine = AutoMLEngine(config)
    engine.fit(X_train, y_train)

    leaderboard = engine.get_leaderboard()
    print('\n📊 AutoML Leaderboard:')
    print(leaderboard.to_string(index=False))

    choice = engine.explain_choice()
    print(f'\n🥇 Best model chosen: {choice["model"]}  (reason: {choice["reason"]})')

else:
    # ── Stub: evaluate a grid of sklearn models manually ──────────────────
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    import time as _time

    candidates = [
        ('RandomForest',        RandomForestClassifier(n_estimators=100, random_state=42)),
        ('GradientBoosting',    GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('LogisticRegression',  LogisticRegression(max_iter=1000, random_state=42)),
        ('SVM',                 SVC(probability=True, random_state=42)),
    ]

    rows = []
    for name, clf in candidates:
        t0 = _time.perf_counter()
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)
        elapsed = _time.perf_counter() - t0
        rows.append({
            'Model':     name,
            'Accuracy':  round(accuracy_score(y_test, preds), 4),
            'F1':        round(f1_score(y_test, preds), 4),
            'Train (s)': round(elapsed, 3),
        })

    leaderboard = pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)
    leaderboard.index += 1
    print('\n📊 AutoML Leaderboard (stub):')
    print(leaderboard.to_string())

    best_row = leaderboard.iloc[0]
    print(f'\n🥇 Best model: {best_row["Model"]}  '
          f'(Accuracy={best_row["Accuracy"]}, F1={best_row["F1"]})')
    print('   Reason: highest accuracy on held-out test set')

## 2. Experiment Tracking

The `ExperimentManager` lets you group multiple model runs into named
experiments, log arbitrary metrics, and retrieve the best run.

In [ ]:
import time as _time

# ── Model configs to compare ───────────────────────────────────────────────
model_configs = [
    ('RF-shallow',  RandomForestModel(task='classification', n_estimators=50,  max_depth=3)),
    ('RF-medium',   RandomForestModel(task='classification', n_estimators=100, max_depth=None)),
    ('RF-deep',     RandomForestModel(task='classification', n_estimators=200, max_depth=None)),
]

def _evaluate_rf(name, model):
    """Train, evaluate and return a metrics dict."""
    t0 = _time.perf_counter()
    model.train(X_train, y_train)
    preds = model.predict(X_test)
    elapsed = _time.perf_counter() - t0
    return {
        'run_name':  name,
        'accuracy':  round(accuracy_score(y_test, preds), 4),
        'f1':        round(f1_score(y_test, preds), 4),
        'precision': round(precision_score(y_test, preds), 4),
        'recall':    round(recall_score(y_test, preds), 4),
        'train_s':   round(elapsed, 3),
    }

if _EXPERIMENTS_AVAILABLE:
    mgr = ExperimentManager()
    exp = mgr.create_experiment('breast-cancer-comparison')

    for name, model in model_configs:
        run = exp.start_run(name)
        metrics = _evaluate_rf(name, model)
        run.log_metrics(metrics)
        run.end()
        print(f'  ✅ Run logged: {name}  accuracy={metrics["accuracy"]}')

    best = exp.get_best_run(metric='accuracy')
    print(f'\n🏆 Best run: {best.name}  (accuracy={best.metrics["accuracy"]})')

    print('\n📋 Full run comparison:')
    print(exp.compare_runs().to_string(index=False))

else:
    # ── Stub: store results in a DataFrame ────────────────────────────────
    run_results = []
    for name, model in model_configs:
        m = _evaluate_rf(name, model)
        run_results.append(m)
        print(f'  ✅ Run logged: {name}  accuracy={m["accuracy"]}')

    runs_df = pd.DataFrame(run_results).set_index('run_name')
    best_run = runs_df['accuracy'].idxmax()

    print(f'\n🏆 Best run: {best_run}  (accuracy={runs_df.loc[best_run, "accuracy"]})')
    print('\n📋 Full run comparison:')
    print(runs_df.to_string())

## 3. Model Explainability

After training we inspect **feature importances** to understand
which inputs drive predictions.

In [ ]:
# ── Train the best RF on full breast-cancer data ───────────────────────────
rf = RandomForestModel(task='classification', n_estimators=200, random_state=42)
rf.train(X_train, y_train)
register_model('rf-breast-cancer', rf)
print(f'✅ Model registered.  Registry: {list_models()}')

if _EXPLAIN_AVAILABLE:
    engine = ExplainabilityEngine(rf)
    importance_dict = engine.feature_importance(X_train, feature_names=list(X.columns))
    fi_series = pd.Series(importance_dict).sort_values(ascending=False)
else:
    # ── Stub: pull importances directly from the sklearn model ────────────
    raw = rf.model.feature_importances_
    fi_series = pd.Series(raw, index=list(X.columns)).sort_values(ascending=False)

# ── Plot top 10 ───────────────────────────────────────────────────────────
top10 = fi_series.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top10)))
bars = ax.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1])

for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Importance Score', fontsize=11)
ax.set_title('Top 10 Feature Importances — Breast Cancer RF', fontsize=13, fontweight='bold')
ax.set_xlim(0, top10.max() * 1.15)
plt.tight_layout()
plt.show()

print('\n🔍 Top 5 features:')
for feat, score in fi_series.head(5).items():
    print(f'   {feat}: {score:.4f}')

## 4. Data Drift Detection

We simulate drift by adding Gaussian noise to a reference dataset
and check which features have shifted beyond the detection threshold.

In [ ]:
from scipy import stats as _stats

# ── Build reference and drifted datasets ──────────────────────────────────
rng = np.random.default_rng(0)
reference = X.copy()

# Drift features 0-4 significantly; leave the rest unchanged
current = reference.copy()
drifted_cols = list(X.columns[:5])
for col in drifted_cols:
    std = reference[col].std()
    current[col] = reference[col] + rng.normal(loc=2.0 * std, scale=0.5 * std, size=len(reference))

print(f'Reference shape : {reference.shape}')
print(f'Current shape   : {current.shape}')
print(f'Intentionally drifted features: {drifted_cols}\n')

if _DRIFT_AVAILABLE:
    detector = DataDriftDetector(threshold=0.05)
    detector.fit_reference(reference)
    report = detector.detect_drift(current)
    print('Drift report:')
    print(pd.DataFrame(report['features']).T.to_string())

else:
    # ── Stub: Kolmogorov-Smirnov test for each numeric feature ────────────
    THRESHOLD = 0.05  # p-value threshold
    results = []
    for col in reference.columns:
        stat, p_value = _stats.ks_2samp(reference[col].dropna(), current[col].dropna())
        results.append({
            'Feature':     col,
            'KS Stat':     round(stat, 4),
            'p-value':     round(p_value, 4),
            'Drift?':      '🔴 YES' if p_value < THRESHOLD else '🟢 NO',
        })

    report_df = pd.DataFrame(results).sort_values('KS Stat', ascending=False).reset_index(drop=True)
    print('📊 Drift Detection Report (KS test):')
    print(report_df.to_string(index=False))

    n_drifted = (report_df['Drift?'] == '🔴 YES').sum()
    print(f'\n🚨 {n_drifted}/{len(report_df)} features show statistically significant drift '
          f'(p < {THRESHOLD})')

## 5. Data Quality

The `DataQualityEngine` supports custom rule definitions and
automatic profiling of a DataFrame.

In [ ]:
# ── Build a realistic sample with quality issues ───────────────────────────
rng2 = np.random.default_rng(7)
n = 300
sample = pd.DataFrame({
    'age':       rng2.integers(18, 80, size=n).astype(float),
    'income':    rng2.exponential(scale=40_000, size=n),
    'score':     rng2.uniform(0, 1, size=n),
    'category':  rng2.choice(['A', 'B', 'C', 'D'], size=n),
    'id_col':    [f'ID-{i:04d}' for i in range(n)],     # high-cardinality
})

# Inject quality issues
sample.loc[rng2.choice(n, 20, replace=False), 'age'] = np.nan
sample.loc[rng2.choice(n, 10, replace=False), 'income'] = np.nan
sample = pd.concat([sample, sample.iloc[:5]], ignore_index=True)  # 5 duplicate rows
sample['const_col'] = 1                                             # zero-variance column

print(f'Sample shape: {sample.shape}')
print(sample.dtypes.to_string(), '\n')

if _QUALITY_AVAILABLE:
    qe = DataQualityEngine()

    # Add domain rules
    qe.add_rule('age_range',    lambda df: df['age'].dropna().between(0, 120).all(),
                'Age must be 0-120')
    qe.add_rule('score_range',  lambda df: df['score'].between(0, 1).all(),
                'Score must be 0-1')
    qe.add_rule('income_pos',   lambda df: (df['income'].dropna() >= 0).all(),
                'Income must be non-negative')

    report = qe.auto_profile(sample)
    print('📋 Data Quality Report:')
    print(report)

else:
    # ── Stub: rule-based quality check ────────────────────────────────────
    class _DataQualityEngineStub:
        def __init__(self):
            self._rules = []

        def add_rule(self, name, fn, description=''):
            self._rules.append((name, fn, description))

        def auto_profile(self, df):
            lines = ['=' * 60, ' DATA QUALITY REPORT', '=' * 60]

            # Shape & memory
            lines.append(f'Rows       : {len(df):,}')
            lines.append(f'Columns    : {len(df.columns)}')
            lines.append(f'Memory     : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

            # Duplicates
            dups = df.duplicated().sum()
            lines.append(f'Duplicates : {dups}  {"⚠️" if dups else "✅"}')

            # Missing
            miss = df.isnull().sum()
            lines.append(f'Missing    : {int(miss.sum())} cells across {int((miss > 0).sum())} columns')
            for col, cnt in miss[miss > 0].items():
                lines.append(f'  ⚠️  {col}: {cnt} missing ({cnt/len(df)*100:.1f}%)')

            # Constant columns
            const = [c for c in df.columns if df[c].nunique() <= 1]
            if const:
                lines.append(f'Constant   : {const}  ⚠️')
            else:
                lines.append('Constant   : none  ✅')

            # High-cardinality
            hi = [c for c in df.select_dtypes(['object', 'category']).columns
                  if df[c].nunique() > 0.5 * len(df)]
            if hi:
                lines.append(f'High-card  : {hi}  ℹ️  (possible IDs)')

            # Custom rules
            if self._rules:
                lines.append('\nCustom Rules:')
                for name, fn, desc in self._rules:
                    try:
                        ok = fn(df)
                        status = '✅ PASS' if ok else '❌ FAIL'
                    except Exception as e:
                        status = f'⚠️  ERROR: {e}'
                    lines.append(f'  [{status}] {name} — {desc}')

            lines.append('=' * 60)
            return '\n'.join(lines)

    qe = _DataQualityEngineStub()
    qe.add_rule('age_range',   lambda df: df['age'].dropna().between(0, 120).all(), 'Age must be 0-120')
    qe.add_rule('score_range', lambda df: df['score'].between(0, 1).all(),          'Score must be 0-1')
    qe.add_rule('income_pos',  lambda df: (df['income'].dropna() >= 0).all(),       'Income must be non-negative')

    report = qe.auto_profile(sample)
    print(report)